In [ ]:
! pip install gensim scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

In [ ]:
# Define the four specific newsgroup classes you want to use
selected_classes = [
    'alt.atheism',
    'comp.graphics',
    'sci.space',
    'talk.politics.mideast'
]

# Load the 20 Newsgroups dataset for the selected classes
newsgroups = fetch_20newsgroups(subset='all', categories=selected_classes, remove=('headers', 'footers', 'quotes'))

In [ ]:
# Preprocess the text data and create a list of tokenized documents
tokenized_documents = [simple_preprocess(text) for text in newsgroups.data]

# Create a dictionary mapping of words to unique IDs
dictionary = corpora.Dictionary(tokenized_documents)

# Create a Bag of Words (BoW) representation of the documents
bow_corpus = [dictionary.doc2bow(doc) for doc in tokenized_documents]

# Train the LDA model
lda_model = models.LdaModel(bow_corpus, num_topics=20, id2word=dictionary, passes=15)

# Print the topics and their top words
topics = lda_model.print_topics(num_words=10)

for topic in topics:
    print(topic)

(0, '0.012*"music" + 0.008*"deaf" + 0.006*"timmons" + 0.006*"tea" + 0.006*"bake" + 0.005*"children" + 0.004*"peter" + 0.004*"yo" + 0.003*"iii" + 0.003*"educ"')
(1, '0.010*"uae" + 0.007*"musa" + 0.006*"caste" + 0.006*"islands" + 0.006*"abu" + 0.004*"iges" + 0.004*"awesome" + 0.003*"trillion" + 0.003*"enhanced" + 0.003*"factions"')
(2, '0.024*"the" + 0.023*"and" + 0.020*"edu" + 0.016*"to" + 0.016*"for" + 0.013*"graphics" + 0.012*"in" + 0.010*"of" + 0.010*"pub" + 0.008*"mail"')
(3, '0.029*"to" + 0.021*"for" + 0.018*"the" + 0.017*"you" + 0.015*"if" + 0.014*"have" + 0.013*"it" + 0.013*"this" + 0.012*"any" + 0.012*"is"')
(4, '0.012*"rw" + 0.012*"bobby" + 0.011*"location" + 0.010*"file" + 0.010*"mar" + 0.009*"host" + 0.008*"zip" + 0.008*"mozumder" + 0.007*"mom" + 0.006*"men"')
(5, '0.090*"the" + 0.054*"of" + 0.033*"and" + 0.028*"in" + 0.021*"to" + 0.012*"by" + 0.010*"armenian" + 0.009*"were" + 0.008*"turkish" + 0.007*"armenians"')
(6, '0.009*"none" + 0.006*"ihr" + 0.006*"smiley" + 0.004*"hija

In [ ]:
# Iterate through the documents and get their topic distributions
document_topic_vectors = []

for doc_bow in bow_corpus:
    document_topics = lda_model.get_document_topics(doc_bow, minimum_probability=0.0)
    document_topic_vector = [topic_prob for _, topic_prob in document_topics]
    document_topic_vectors.append(document_topic_vector)

In [ ]:
# Specify the number of documents to print
num_documents_to_print = 5

# Iterate through the documents and get their topic distributions
document_topic_vectors = []
for i, doc_bow in enumerate(bow_corpus):
    document_topics = lda_model.get_document_topics(doc_bow, minimum_probability=0.0)
    document_topic_vector = [topic_prob for _, topic_prob in document_topics]
    document_topic_vectors.append(document_topic_vector)

    # Print the topic vector for the first num_documents_to_print documents
    if i < num_documents_to_print:
        print(f"Document {i + 1} Topic Vector: {document_topic_vector}")

Document 1 Topic Vector: [np.float32(0.003848985), np.float32(0.003848985), np.float32(0.09417852), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.0038489853), np.float32(0.003848985), np.float32(0.15742785), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.68296075), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.003848985), np.float32(0.003848985)]
Document 2 Topic Vector: [np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(0.99907166), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32(4.886989e-05), np.float32

In [ ]:
num_topics = 20
alphas = [0.01, 1.0, 2.0, 'auto']
etas = [0.01, 1.0, 2.0, 'auto']

results = []

for alpha in alphas:
    for eta in etas:
        print("\nalpha =", alpha, "eta =", eta)

        lda_model = models.LdaModel(
            corpus=bow_corpus,
            id2word=dictionary,
            num_topics=num_topics,
            passes=15,
            alpha=alpha,
            eta=eta,
            random_state=42
        )

        # coherence score
        coherence_model = CoherenceModel(
            model=lda_model,
            texts=tokenized_documents,
            dictionary=dictionary,
            coherence='c_v'
        )

        coherence_score = coherence_model.get_coherence()

        perplexity_score = lda_model.log_perplexity(bow_corpus)

        print(f"\nCoherence Score : {coherence_score:.4f}")
        print(f"Log Perplexity  : {perplexity_score:.4f}")

        print("\nTopics:")

        topics = lda_model.print_topics(num_words=7)

        for topic in topics:
            print(topic)

        print("\nFirst document topic vectors:")

        for i in range(3):

            doc_topics = lda_model.get_document_topics(
                bow_corpus[i],
                minimum_probability=0.0
            )

            topic_vector = [prob for _, prob in doc_topics]

            print(f"\nDocument {i+1}:")
            print(topic_vector)

        results.append({
            'alpha': alpha,
            'eta': eta,
            'coherence': coherence_score,
            'log_perplexity': perplexity_score
        })


alpha = 0.01 eta = 0.01

Coherence Score : 0.3619
Log Perplexity  : -9.9819

Topics:
(0, '0.023*"istanbul" + 0.020*"of" + 0.020*"the" + 0.019*"by" + 0.017*"ve" + 0.014*"ankara" + 0.013*"ed"')
(1, '0.031*"the" + 0.028*"space" + 0.019*"and" + 0.018*"nasa" + 0.015*"on" + 0.015*"of" + 0.013*"that"')
(2, '0.075*"ed" + 0.019*"of" + 0.018*"the" + 0.017*"that" + 0.017*"bobby" + 0.012*"bus" + 0.012*"and"')
(3, '0.041*"the" + 0.026*"and" + 0.024*"to" + 0.023*"of" + 0.020*"is" + 0.017*"for" + 0.013*"in"')
(4, '0.073*"the" + 0.036*"to" + 0.028*"of" + 0.025*"and" + 0.017*"in" + 0.017*"is" + 0.015*"for"')
(5, '0.099*"the" + 0.047*"of" + 0.036*"in" + 0.026*"and" + 0.026*"to" + 0.011*"was" + 0.010*"by"')
(6, '0.074*"the" + 0.036*"of" + 0.028*"and" + 0.021*"to" + 0.019*"in" + 0.010*"on" + 0.009*"for"')
(7, '0.055*"the" + 0.036*"of" + 0.032*"to" + 0.031*"is" + 0.025*"that" + 0.020*"and" + 0.016*"it"')
(8, '0.044*"you" + 0.042*"the" + 0.036*"to" + 0.027*"that" + 0.022*"is" + 0.022*"it" + 0.018*"and"')
(